# Deep Q-Network (DQN)

¡Bienvenido a la tarea de DQN! Aquí es donde el Reinforcement Learning se encuentra con el Deep Learning. Al final de este notebook, podrás:

* Entender por qué necesitamos aproximación de funciones en RL
* Implementar una Red Neuronal para la aproximación de Q-values
* Construir un buffer de experiencia (replay buffer)
* Implementar el algoritmo DQN con redes target
* Entrenar un agente en ambientes con espacios de estados continuos

## De Q-Learning a DQN

**Problema con Q-Tables:**
- CartPole tiene ~10^20 estados posibles (continuos)
- Atari tiene 256^(84×84×4) ≈ 10^67,000 estados!
- Las tablas Q son imposibles para espacios de estados grandes/continuos

**Solución: Aproximación de Funciones**
- En lugar de una tabla, usamos una Red Neuronal: $Q(s,a; \theta)$
- La red aprende a **aproximar** los Q-values
- ¡Puede generalizar a estados nunca vistos!

## Innovaciones Clave de DQN

1. **Red Neuronal**: Reemplaza la Q-table con una Red Neuronal
2. **Experience Replay**: Almacena y reutiliza experiencias pasadas
3. **Red Target**: Estabiliza el entrenamiento con una red Q separada para los objetivos
4. **Reward Clipping**: Normaliza las recompensas para mayor estabilidad

<img src="https://miro.medium.com/max/1400/1*w5GuxedZ9ivRYhQCv8kVZQ.png" style="width:600px;height:300px;">

## Nota Importante sobre la Entrega

Por favor asegúrate de:
1. No incluir sentencias print extra
2. No incluir celdas de código extra  
3. No cambiar los parámetros de las funciones
4. No usar variables globales en las funciones calificadas

## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Arquitectura de Q-Network](#2)
    - [Ejercicio 1 - QNetwork](#ex-1)
- [3 - Buffer de Experiencia (Experience Replay)](#3)
    - [Ejercicio 2 - ReplayBuffer](#ex-2)
- [4 - Entrenamiento DQN](#4)
    - [Ejercicio 3 - compute_td_loss](#ex-3)
    - [Ejercicio 4 - train_dqn](#ex-4)
- [5 - Prueba en CartPole](#5)

<a name='1'></a>
## 1 - Paquetes

In [ ]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import deque, namedtuple
from dqn_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 6.0)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

np.random.seed(42)
torch.manual_seed(42)

<a name='2'></a>
## 2 - Arquitectura de Q-Network

La Q-Network es una red neuronal que toma un estado como entrada y devuelve Q-values para cada acción.

**Arquitectura para CartPole:**
```
Input: Estado (4 valores) → [posición carro, velocidad carro, ángulo palo, velocidad angular palo]
Capa Oculta 1: 128 neuronas (ReLU)
Capa Oculta 2: 128 neuronas (ReLU)
Output: Q-values (2 acciones) → [Q(s, izquierda), Q(s, derecha)]
```

<a name='ex-1'></a>
### Ejercicio 1 - QNetwork

Implementa una Q-Network usando PyTorch.

In [ ]:
# GRADED FUNCTION: QNetwork

class QNetwork(nn.Module):
    """
    Red Q (Q-Network) para DQN.
    
    Argumentos:
    state_dim -- dimensión del espacio de estados
    action_dim -- dimensión del espacio de acciones
    hidden_dim -- número de neuronas en capas ocultas (default: 128)
    """
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(QNetwork, self).__init__()
        
        # (approx. 5-7 lines)
        # Construye una Red Neuronal con:
        # 1. Capa de entrada: state_dim
        # 2. Capa oculta 1: Linear(state_dim, hidden_dim) + ReLU
        # 3. Hidden layer 2: Linear(hidden_dim, hidden_dim) + ReLU
        # 4. Capa de salida: Linear(hidden_dim, action_dim)
        # 
        # Pista: Usa nn.Sequential to combine layers
        # Hint: ReLU activation: nn.ReLU()
        
        # TU CÓDIGO EMPIEZA AQUÍ
        
        
        
        
        
        # TU CÓDIGO TERMINA AQUÍ
    
    def forward(self, state):
        """
        Paso hacia adelante (Forward pass).
        
        Argumentos:
        state -- tensor de estado de forma (batch_size, state_dim)
        
        Returns:
        q_values -- Q-values para cada acción, shape (batch_size, action_dim)
        """
        # (approx. 1 line)
        # Pasa el estado a través de la red
        
        # TU CÓDIGO EMPIEZA AQUÍ
        
        
        # TU CÓDIGO TERMINA AQUÍ
        
        return q_values

In [ ]:
# Prueba tu implementación
q_net = QNetwork(state_dim=4, action_dim=2, hidden_dim=128)
print("Q-Network architecture:")
print(q_net)

# Prueba forward pass
test_state = torch.randn(1, 4)
q_values = q_net(test_state)
print(f"\nInput shape: {test_state.shape}")
print(f"Output shape: {q_values.shape}")
print(f"Q-values: {q_values.detach().numpy()}")

# Run the grader
qnetwork_test(QNetwork)

<a name='3'></a>
## 3 - Buffer de Experiencia (Experience Replay)

**¿Por qué Experience Replay?**

Problema con el aprendizaje online:
- Las muestras consecutivas están altamente correlacionadas
- La red puede olvidar experiencias previas (olvido catastrófico)
- Uso ineficiente de los datos

**Solución: Replay Buffer**
1. Almacenar experiencias (s, a, r, s', done) en un buffer
2. Muestrear mini-batches aleatorios para el entrenamiento
3. Rompe la correlación, mejora la estabilidad y la eficiencia de las muestras

```python
# Guardar experiencia
buffer.add(state, action, reward, next_state, done)

# Muestrear mini-batch
batch = buffer.sample(batch_size=32)
```

<a name='ex-2'></a>
### Ejercicio 2 - ReplayBuffer

Implementa un buffer de experiencia.

In [ ]:
# Define experience tuple
Experience = namedtuple('Experience', ['state', 'action', 'reward', 'next_state', 'done'])

In [ ]:
# GRADED FUNCTION: ReplayBuffer

class ReplayBuffer:
    """
    Buffer de Experiencia (Replay Buffer).
    
    Argumentos:
    capacity -- número máximo de experiencias a almacenar
    """
    
    def __init__(self, capacity=10000):
        # (approx. 1 line)
        # Usa deque con maxlen=capacity to store experiences
        # Hint: deque elimina automáticamente los ítems más viejos when full
        
        # TU CÓDIGO EMPIEZA AQUÍ
        
        
        # TU CÓDIGO TERMINA AQUÍ
    
    def add(self, state, action, reward, next_state, done):
        """
        Añadir experiencia al buffer.
        """
        # (approx. 1-2 lines)
        # Crear Experience tuple y append to buffer
        
        # TU CÓDIGO EMPIEZA AQUÍ
        
        
        # TU CÓDIGO TERMINA AQUÍ
    
    def sample(self, batch_size):
        """
        Muestrear un lote aleatorio de experiencias.
        
        Argumentos:
        batch_size -- número de experiencias a muestrear
        
        Returns:
        Tupla de lotes (batch) (states, actions, rewards, next_states, dones)
        """
        # (approx. 8-10 lines)
        # 1. Muestrear aleatoriamente batch_size experiences from buffer
        #    Hint: Use np.random.choice to sample indices
        # 2. Extraer y apilar cada componente into numpy arrays
        # 3. Retornar tupla de (states, actions, rewards, next_states, dones)
        
        # TU CÓDIGO EMPIEZA AQUÍ
        
        
        
        
        
        
        
        # TU CÓDIGO TERMINA AQUÍ
        
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        """Return current size of buffer."""
        return len(self.buffer)

In [ ]:
# Prueba tu implementación
buffer = ReplayBuffer(capacity=1000)

# Add some experiences
for i in range(10):
    state = np.array([i, i+1, i+2, i+3])
    action = i % 2
    reward = float(i)
    next_state = state + 1
    done = (i == 9)
    buffer.add(state, action, reward, next_state, done)

print(f"Buffer size: {len(buffer)}")

# Sample a batch
states, actions, rewards, next_states, dones = buffer.sample(batch_size=3)
print(f"\nSampled batch shapes:")
print(f"States: {states.shape}")
print(f"Actions: {actions.shape}")
print(f"Rewards: {rewards.shape}")

# Run the grader
replay_buffer_test(ReplayBuffer)

<a name='4'></a>
## 4 - Entrenamiento DQN

DQN utiliza **dos redes**:
1. **Q-Network** (θ): Actualizada en cada paso, usada para seleccionar acciones
2. **Target Network** (θ⁻): Actualizada periódicamente, usada para calcular los objetivos (targets)

**¿Por qué Target Network?**
- Previene el problema del objetivo móvil (moving target problem)
- Estabiliza el entrenamiento
- Se actualiza cada C pasos: θ⁻ ← θ

**Función de Loss de DQN:**
$$L(\theta) = \mathbb{E}_{(s,a,r,s')\sim Buffer}\left[\left(r + \gamma \max_{a'} Q(s',a'; \theta^-) - Q(s,a;\theta)\right)^2\right]$$

<a name='ex-3'></a>
### Ejercicio 3 - compute_td_loss

Implementa el cálculo de la loss para DQN.

In [ ]:
# GRADED FUNCTION: compute_td_loss

def compute_td_loss(q_network, target_network, batch, gamma=0.99):
    """
    Calcular la pérdida TD para DQN.
    
    Argumentos:
    q_network -- red Q actual
    target_network -- red Q objetivo (target)  
    batch -- tuple of (states, actions, rewards, next_states, dones)
    gamma -- discount factor
    
    Returns:
    loss -- error cuadrático medio TD
    """
    states, actions, rewards, next_states, dones = batch
    
    # Convertir a tensores (approx. 5 lines)
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    
    
    # TU CÓDIGO TERMINA AQUÍ
    
    # Calcular Q-values (approx. 5-7 lines)
    # 1. Obtener Q-values actuales: Q(s, a) for acciones tomadas
    #    Hint: Use gather() to select Q-values for specific actions
    # 2. Obtener Q-values objetivo: max Q(s', a') from target network
    #    Hint: Use torch.no_grad() for target network
    # 3. Calcular objetivos TD: r + gamma * max Q(s', a') * (1 - done)
    # 4. Calcular pérdida (loss): MSE entre Q actual y TD targets
    
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    
    
    
    
    
    
    # TU CÓDIGO TERMINA AQUÍ
    
    return loss

<a name='4'></a>
## 4 - Entrenamiento DQN

DQN utiliza **dos redes**:
1. **Q-Network** (θ): Actualizada en cada paso, usada para seleccionar acciones
2. **Target Network** (θ⁻): Actualizada periódicamente, usada para calcular los objetivos (targets)

**¿Por qué Target Network?**
- Previene el problema del objetivo móvil (moving target problem)
- Estabiliza el entrenamiento
- Se actualiza cada C pasos: θ⁻ ← θ

**Función de Loss de DQN:**
$$L(\theta) = \mathbb{E}_{(s,a,r,s')\sim Buffer}\left[\left(r + \gamma \max_{a'} Q(s',a'; \theta^-) - Q(s,a;\theta)\right)^2\right]$$

<a name='ex-3'></a>
### Ejercicio 3 - compute_td_loss

Implementa el cálculo de la loss para DQN.

In [ ]:
# Helper function
def select_action(q_network, state, epsilon, n_actions):
    """Epsilon-greedy action selection."""
    if np.random.random() < epsilon:
        return np.random.randint(n_actions)
    else:
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = q_network(state_tensor)
            return q_values.argmax().item()

In [ ]:
# GRADED FUNCTION: train_dqn

def train_dqn(env, n_episodes=500, batch_size=32, gamma=0.99,
              epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
              lr=0.001, target_update=10, buffer_size=10000):
    """
    Entrenar agente DQN.
    
    Returns:
    q_network -- red Q entrenada
    rewards_history -- lista de recompensas por episodio
    """
    # Inicializar (approx. 8 lines)
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    
    
    
    
    # TU CÓDIGO TERMINA AQUÍ
    
    rewards_history = []
    epsilon = epsilon_start
    
    # Bucle de entrenamiento (approx. 25-30 lines)
    # Para cada episodio:
    #   1. Resetear ambiente
    #   2. Para cada paso:
    #      a. Seleccionar acción usando epsilon-greedy
    #      b. Tomar acción, observar recompensa y next_state
    #      c. Guardar experiencia en buffer
    #      d. Si el buffer tiene suficientes muestras:
    #         - Muestrear batch del buffer
    #         - Calcular pérdida (loss)
    #         - Actualizar red Q
    #      e. Si terminó (done), romper ciclo
    #   3. Decaer epsilon
    #   4. Cada target_update episodios, copiar red Q a red target
    #   5. Guardar recompensa del episodio
    
    # TU CÓDIGO EMPIEZA AQUÍ
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    # TU CÓDIGO TERMINA AQUÍ
    
    return q_network, rewards_history

<a name='5'></a>
## 5 - Prueba en CartPole

¡Vamos a entrenar DQN en el ambiente clásico CartPole!

**Objetivo**: Balancear un palo sobre un carro el mayor tiempo posible.
- **Estado**: [posición carro, velocidad carro, ángulo palo, velocidad angular palo]
- **Acciones**: [empujar izquierda, empujar derecha]
- **Recompensa**: +1 por cada instante de tiempo que el palo se mantiene arriba
- **Éxito**: Recompensa promedio ≥ 195 sobre 100 episodios

In [ ]:
# Crear environment
env = gym.make('CartPole-v1')

print("Entrenando DQN en CartPole...")
print(f"State space: {env.observation_space.shape[0]}")
print(f"Action space: {env.action_space.n}")
print("\nEsto puede tomar unos minutos...\n")

# Entrena
q_network, rewards_history = train_dqn(
    env,
    n_episodes=500,
    batch_size=32,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995,
    lr=0.001,
    target_update=10,
    buffer_size=10000
)

print("\n¡Entrenamiento completado!")
print(f"Recompensa promedio final (últimos 100 eps): {np.mean(rewards_history[-100:]):.2f}")

In [ ]:
# Plot results
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(rewards_history, alpha=0.3, label='Recompensa del episodio')

window = 20
if len(rewards_history) >= window:
    moving_avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(rewards_history)), moving_avg,
            label=f'Promedio móvil ({window} episodes)', linewidth=2)

ax.axhline(y=195, color='r', linestyle='--', label='Umbral de éxito (195)', alpha=0.7)
ax.set_xlabel('Episodio')
ax.set_ylabel('Recompensa Total')
ax.set_title('Entrenamiento DQN en CartPole')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

env.close()

# Deep Q-Network (DQN)

¡Bienvenido a la tarea de DQN! Aquí es donde el Reinforcement Learning se encuentra con el Deep Learning. Al final de este notebook, podrás:

* Entender por qué necesitamos aproximación de funciones en RL
* Implementar una Red Neuronal para la aproximación de Q-values
* Construir un buffer de experiencia (replay buffer)
* Implementar el algoritmo DQN con redes target
* Entrenar un agente en ambientes con espacios de estados continuos

## De Q-Learning a DQN

**Problema con Q-Tables:**
- CartPole tiene ~10^20 estados posibles (continuos)
- Atari tiene 256^(84×84×4) ≈ 10^67,000 estados!
- Las tablas Q son imposibles para espacios de estados grandes/continuos

**Solución: Aproximación de Funciones**
- En lugar de una tabla, usamos una Red Neuronal: $Q(s,a; \theta)$
- La red aprende a **aproximar** los Q-values
- ¡Puede generalizar a estados nunca vistos!

## Innovaciones Clave de DQN

1. **Red Neuronal**: Reemplaza la Q-table con una Red Neuronal
2. **Experience Replay**: Almacena y reutiliza experiencias pasadas
3. **Red Target**: Estabiliza el entrenamiento con una red Q separada para los objetivos
4. **Reward Clipping**: Normaliza las recompensas para mayor estabilidad

<img src="https://miro.medium.com/max/1400/1*w5GuxedZ9ivRYhQCv8kVZQ.png" style="width:600px;height:300px;">

## Nota Importante sobre la Entrega

Por favor asegúrate de:
1. No incluir sentencias print extra
2. No incluir celdas de código extra  
3. No cambiar los parámetros de las funciones
4. No usar variables globales en las funciones calificadas